In [7]:

import os
import sys
from pathlib import Path

# Find the project root
PROJECT_ROOT = Path.cwd()

# If notebook is running inside notebooks/, go one level up
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Add project root to Python import path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("src folder exists:", (PROJECT_ROOT / "src").exists())


Project root: /Users/mubhinbasnet/Desktop/student-dropout-prediction
src folder exists: True


In [8]:

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)



In [9]:

import importlib
import src.data

importlib.reload(src.data)
from src.data import create_data_split
from src.features import build_preprocessor
from src.config import RANDOM_STATE

# Load the original split
X_train, X_test, y_train, y_test = create_data_split()

# Build the selected model
final_model = Pipeline([
    ("preprocessor", build_preprocessor()),
    ("classifier", LogisticRegression(
        C=1.0,
        class_weight=None,
        max_iter=2000,
        random_state=RANDOM_STATE
    ))
])

# Train using the complete training dataset
final_model.fit(X_train, y_train)

print("Final model trained successfully!")

Dataset successfully prepared
----------------------------------------
Total records: 3630
Features: 24

Training records: 2904
Testing records: 726

Training target distribution:
target
0    0.608
1    0.392
Name: proportion, dtype: float64

Testing target distribution:
target
0    0.609
1    0.391
Name: proportion, dtype: float64
Final model trained successfully!


In [10]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

# Generate predictions on the untouched test set
y_test_pred = final_model.predict(X_test)
y_test_proba = final_model.predict_proba(X_test)[:, 1]

# Calculate evaluation metrics
test_results = {
    "Accuracy": accuracy_score(y_test, y_test_pred),
    "Precision": precision_score(y_test, y_test_pred),
    "Recall": recall_score(y_test, y_test_pred),
    "F1 Score": f1_score(y_test, y_test_pred),
    "ROC-AUC": roc_auc_score(y_test, y_test_proba),
    "PR-AUC": average_precision_score(y_test, y_test_proba)
}

print("FINAL MODEL TEST RESULTS")
print("------------------------")

for metric, value in test_results.items():
    print(f"{metric}: {value:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_test_pred,
    target_names=["Graduate", "Dropout"]
))

FINAL MODEL TEST RESULTS
------------------------
Accuracy: 0.7837
Precision: 0.7510
Recall: 0.6690
F1 Score: 0.7076
ROC-AUC: 0.8535
PR-AUC: 0.8201

Confusion Matrix:
[[379  63]
 [ 94 190]]

Classification Report:
              precision    recall  f1-score   support

    Graduate       0.80      0.86      0.83       442
     Dropout       0.75      0.67      0.71       284

    accuracy                           0.78       726
   macro avg       0.78      0.76      0.77       726
weighted avg       0.78      0.78      0.78       726

